<a href="https://colab.research.google.com/github/Dharshini1701/priyadharshini/blob/main/cleaning_15000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas openpyxl

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving orders_staging_15000.xlsx to orders_staging_15000.xlsx


In [ ]:
import pandas as pd
df=pd.read_excel("orders_staging_15000.xlsx")
print(df.columns)

Index(['OrderID', 'CustomerName', 'Email', 'Phone', 'City', 'OrderDate',
       'Amount', 'Status', 'PaymentMode', 'SalesChannel'],
      dtype='object')


In [ ]:
import sqlite3
conn=sqlite3.connect("orders.db")
cursor=conn.cursor()

In [ ]:
df.to_sql("orders_staging",conn,if_exists="replace",index=False)

15000

In [ ]:
cursor.execute("select * from orders_staging limit 5")
for row in cursor.fetchall():
 print(row)

('ORD100001', 'Anitha Singh', 'anitha.singh861@outlook.com', '7469267437', 'Erode', '2024-05-10', '1274.08', 'Pending', 'NetBanking', 'Store')
('ORD100002', 'Suresh Raj', 'suresh.raj5312@company.com', '4095809263', 'Bangalore', '2024-05-10', '43032.04', 'Returned', 'UPI', 'Mobile App')
('ORD100003', 'Hari Singh', 'hari.singh8793@company.com', '1989413672', 'Chennai', '2025-07-15', '33210.48', 'Pending', 'Card', 'Marketplace')
('ORD100004', 'Hari Krishnan', 'hari.krishnan5676@yahoo.com', '9376874147', 'Hyderabad', '2024-08-04', '14226.49', 'Shipped', 'Wallet', 'Online')
('ORD100005', 'Sathish Kumar', 'sathish.kumar8007@gmail.com', '7077207220', 'Salem', '2025-05-04', '18083.67', 'Completed', 'NetBanking', 'Mobile App')


In [ ]:
cursor.execute(""" select count(*)-count(OrderID) as OrderID_null,
count(*)-count(OrderDate) as OrderDate_null,
count(*)-count(CustomerName) as CustomerName_null,
count(*)-count(Email)as Email_null,
count(*)-count(Phone)as Phone_null,
count(*)-count(SalesChannel)as salesChannel_null,
count(*)-count(City)as City_null,
count(*)-count(PaymentMode)as PaymentMode_null,
count(*)-count(Amount)as Amount_null,
count(*)-(status)as status_null from orders_staging""")
print(cursor.fetchall())

[(0, 100, 50, 100, 100, 50, 100, 100, 100, 15000)]


In [ ]:
cursor.execute(""" select orderID,CustomerName,Email,Phone,SalesChannel ,City,PaymentMode,Amount,Status,count(*) from orders_staging group by OrderID,CustomerName,Email,OrderDate,Phone,SalesChannel,City,PaymentMode,Amount,Status having count(*)>1""")
for row in cursor.fetchall():
  print(row)

In [ ]:
cursor.execute("""create table orders_clean as select distinct * from orders_staging""")

In [ ]:
cursor.execute(""" select CustomerName,
trim(CustomerName)as trim_name,
lower(CustomerName)as lower_name,
upper(customerName)as upper_name,
substr(CustomerName(1,3),
instr(customerName(,''),
CustomerName||_'Customer' from orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)

In [ ]:
cursor.execute("""
select Email,lower(trim(Email)), case
when lower(trim(Email)) like '%@%.%' then 'valid'else 'invalid' end as Email_invalid from orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)

('anitha.singh861@outlook.com', 'anitha.singh861@outlook.com', 'valid')
('suresh.raj5312@company.com', 'suresh.raj5312@company.com', 'valid')
('hari.singh8793@company.com', 'hari.singh8793@company.com', 'valid')
('hari.krishnan5676@yahoo.com', 'hari.krishnan5676@yahoo.com', 'valid')
('sathish.kumar8007@gmail.com', 'sathish.kumar8007@gmail.com', 'valid')
('divya.reddy3105@company.com', 'divya.reddy3105@company.com', 'valid')
('deepa.das6939@gmail.com', 'deepa.das6939@gmail.com', 'valid')
('ravi.krishnan9790@gmail.com', 'ravi.krishnan9790@gmail.com', 'valid')
('meena.kumar664@outlook.com', 'meena.kumar664@outlook.com', 'valid')
('ravi.raj855@gmail.com', 'ravi.raj855@gmail.com', 'valid')


In [ ]:
cursor.execute(""" select Phone,
replace(replace(replace(Phone,'+',''),'-',''),'','') from orders_staging limit 10""")
for row in cursor.fetchall():
 print(row)

('7469267437', '7469267437')
('4095809263', '4095809263')
('1989413672', '1989413672')
('9376874147', '9376874147')
('7077207220', '7077207220')
('6674275202', '6674275202')
('9926033466', '9926033466')
('4539686008', '4539686008')
('5783009361', '5783009361')
('0021495636', '0021495636')


In [ ]:
cursor.execute("""select Phone,case
when Phone is not Null and length(phone)=10
and Phone not glob '*[^0-9]*' then 'valid' else 'invalid'
end as Phone_invalid from orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)

('7469267437', 'valid')
('4095809263', 'valid')
('1989413672', 'valid')
('9376874147', 'valid')
('7077207220', 'valid')
('6674275202', 'valid')
('9926033466', 'valid')
('4539686008', 'valid')
('5783009361', 'valid')
('0021495636', 'valid')


In [ ]:
cursor.execute(""" select City,
upper(substr(trim(City),1,1))||
lower(substr(trim(City),2)) from orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)

('Erode', 'Erode')
('Bangalore', 'Bangalore')
('Chennai', 'Chennai')
('Hyderabad', 'Hyderabad')
('Salem', 'Salem')
('salem', 'Salem')
('Trichy', 'Trichy')
('Bangalore', 'Bangalore')
('Madurai', 'Madurai')
('Erode', 'Erode')


In [ ]:
cursor.execute("""
select
OrderDate,
case
when OrderDate like'____-__-__' then OrderDate
when OrderDate like'__-__-____' then substr(OrderDate,7,4)||'-'||
substr(OrderDate,4,2)||'-'||
substr(OrderDate,1,2)
when OrderDate like'__/__/____' then
substr(OrderDate,7,4)||'-'||
substr(OrderDate,4,2)||'-'||
substr(OrderDate,1,2)
else null end as clean_OrderDate from Orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)

('2024-05-10', '2024-05-10')
('2024-05-10', '2024-05-10')
('2025-07-15', '2025-07-15')
('2024-08-04', '2024-08-04')
('2025-05-04', '2025-05-04')
('2025-04-06', '2025-04-06')
('06-08-2025', '2025-08-06')
('2025-09-19', '2025-09-19')
('2024-04-22', '2024-04-22')
('05-02-2024', '2024-02-05')


In [ ]:
cursor.execute("""
select Amount,cast(
replace(replace(replace(Amount,'₹',''),
',',''),
'','')as real)as Amount_clean from orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)

('1274.08', 1274.08)
('43032.04', 43032.04)
('33210.48', 33210.48)
('14226.49', 14226.49)
('18083.67', 18083.67)
('ABC', 0.0)
('4079.75', 4079.75)
('17790.3', 17790.3)
('7617.32', 7617.32)
('12297.46', 12297.46)


In [ ]:
cursor.execute("""
select Amount,case
when Amount is null or trim(Amount)='' then 'missing'
when Amount='ABC' then 'invalid'
when cast(replace(replace(replace(Amount,'₹',''),',',''),' ','')as real)<0 then 'negative'
when cast(replace(replace(replace(Amount,'₹',''),',',''),' ','')as real)=0 then 'zero'
else 'ok' end as Amount_flag from orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)




('1274.08', 'ok')
('43032.04', 'ok')
('33210.48', 'ok')
('14226.49', 'ok')
('18083.67', 'ok')
('ABC', 'invalid')
('4079.75', 'ok')
('17790.3', 'ok')
('7617.32', 'ok')
('12297.46', 'ok')


In [ ]:
cursor.execute("""
select
Status,upper(trim(Status)),
PaymentMode,upper(trim(PaymentMode)),
SalesChannel,upper(trim(SalesChannel))
from orders_staging limit 10""")
for row in cursor.fetchall():
  print(row)

('Pending', 'PENDING', 'NetBanking', 'NETBANKING', 'Store', 'STORE')
('Returned', 'RETURNED', 'UPI', 'UPI', 'Mobile App', 'MOBILE APP')
('Pending', 'PENDING', 'Card', 'CARD', 'Marketplace', 'MARKETPLACE')
('Shipped', 'SHIPPED', 'Wallet', 'WALLET', 'Online', 'ONLINE')
('Completed', 'COMPLETED', 'NetBanking', 'NETBANKING', 'Mobile App', 'MOBILE APP')
('Returned', 'RETURNED', 'Card', 'CARD', 'Mobile App', 'MOBILE APP')
('Cancelled', 'CANCELLED', 'Cash', 'CASH', 'Store', 'STORE')
('Cancelled', 'CANCELLED', 'Cash', 'CASH', 'Online', 'ONLINE')
('Completed', 'COMPLETED', 'Cash', 'CASH', 'Online', 'ONLINE')
('Returned', 'RETURNED', 'NetBanking', 'NETBANKING', 'Store', 'STORE')


In [ ]:
cursor.execute("""
select* from orders_clean limit 5""")
for row in cursor.fetchall():
  print(row)

('ORD100001', 'Anitha Singh', 'anitha.singh861@outlook.com', '7469267437', 'Erode', '2024-05-10', '1274.08', 'Pending', 'NetBanking', 'Store')
('ORD100002', 'Suresh Raj', 'suresh.raj5312@company.com', '4095809263', 'Bangalore', '2024-05-10', '43032.04', 'Returned', 'UPI', 'Mobile App')
('ORD100003', 'Hari Singh', 'hari.singh8793@company.com', '1989413672', 'Chennai', '2025-07-15', '33210.48', 'Pending', 'Card', 'Marketplace')
('ORD100004', 'Hari Krishnan', 'hari.krishnan5676@yahoo.com', '9376874147', 'Hyderabad', '2024-08-04', '14226.49', 'Shipped', 'Wallet', 'Online')
('ORD100005', 'Sathish Kumar', 'sathish.kumar8007@gmail.com', '7077207220', 'Salem', '2025-05-04', '18083.67', 'Completed', 'NetBanking', 'Mobile App')


In [ ]:
cursor.execute("select count(*) from orders_staging")
print("staging:",cursor.fetchone()[0])
cursor.execute("select count(*) from orders_clean")
print("clean:",cursor.fetchone()[0])


staging: 15000
clean: 15000


In [ ]:
cursor.execute("""
select count(*) from (select OrderID,count(*) from orders_clean group by OrderID having count(*)>1)""")
print("duplicates:",cursor.fetchone()[0])

duplicates: 0


In [ ]:
cursor.execute(""" select
count(*)-count(CustomerName),
count(*)-count(Email),
count(*)-count(Phone),
count(*)-count(OrderDate),
count(*)-count(Amount) from orders_clean""")
print("NULL counts:", cursor.fetchone())

NULL counts: (50, 100, 100, 100, 100)


In [ ]:
cursor.execute("""
ALTER TABLE orders_clean
ADD COLUMN Phone_Valid TEXT
""")

In [ ]:
cursor.execute("""
UPDATE orders_clean
SET Phone_Valid =
CASE
    WHEN Phone IS NOT NULL
    AND LENGTH(Phone) = 10
    AND Phone NOT GLOB '*[^0-9]*'
    THEN 'valid'
    ELSE 'invalid'
END
""")

conn.commit()

In [ ]:
cursor.execute("""select Phone_valid,count(*) from orders_clean group by Phone_valid""")
for row in cursor.fetchall():
  print(row)

('invalid', 1000)
('valid', 14000)


In [ ]:
cursor.execute("SELECT * FROM orders_clean LIMIT 5")

for row in cursor.fetchall():
    print(row)

('ORD100001', 'Anitha Singh', 'anitha.singh861@outlook.com', '7469267437', 'Erode', '2024-05-10', '1274.08', 'Pending', 'NetBanking', 'Store', 'valid')
('ORD100002', 'Suresh Raj', 'suresh.raj5312@company.com', '4095809263', 'Bangalore', '2024-05-10', '43032.04', 'Returned', 'UPI', 'Mobile App', 'valid')
('ORD100003', 'Hari Singh', 'hari.singh8793@company.com', '1989413672', 'Chennai', '2025-07-15', '33210.48', 'Pending', 'Card', 'Marketplace', 'valid')
('ORD100004', 'Hari Krishnan', 'hari.krishnan5676@yahoo.com', '9376874147', 'Hyderabad', '2024-08-04', '14226.49', 'Shipped', 'Wallet', 'Online', 'valid')
('ORD100005', 'Sathish Kumar', 'sathish.kumar8007@gmail.com', '7077207220', 'Salem', '2025-05-04', '18083.67', 'Completed', 'NetBanking', 'Mobile App', 'valid')
